 <div class="alert alert-block alert-success">
      <H1><center>Homework</center></H1>

 <div class="alert alert-block alert-danger">
<H4>In the regular semester, I would have you write all the code for this assignment, however given the compressed nature of the summer session, I have provided all code for you. I want you to focus on interpretation this summer.</H4>

# Homework Part 1: Interpreting Data- Morgan Fingerprints

Turn in Part 1 as a written assignment. 

1) Run the statins molecules using extended connectivity fingerprints in the code cells below. The current code sets the ECFP equivalent radius value to 1 with 1024 bits. Compare the Tanimoto similarity values to the MACCS data found previously. Explain what happens to the similarity values.
2) Rerun this code with three different radius setting 0, 1 and 2. (You just ran radius 1 for question 1.)
- Calculate pairwise Tanimoto similarity scores for the same set of molecules at each radius setting.
- Compare how the similarity scores change as the radius increases.
- Describe any trends you observe. For example:
    - do similarity scores tend to increase or decrease as radius increases or decreases?
    - are the molecules ranked similarly or differently across radii?
- Explain why you think these changes happen. Consider what kinds of molecular features are being captured at each radius. Hint:Think about what a radius of 0, 1, or 2 means in terms of what the fingerprint is seeing. How much of the molecular neighborhood around each atom is being encoded?

In [ ]:
import requests
import time
from rdkit import Chem
from rdkit.Chem import Draw


In [ ]:
cids = [    54454,  # Simvastatin (Zocor)              
            54687,  # Pravastatin (Pravachol)
            60823,  # Atorvastatin (Lipitor)
           446155,  # Fluvastatin (Lescol)   
           446157,  # Rosuvastatin (Crestor)
          5282452,  # Pitavastatin (Livalo)
         97938126 ] # Lovastatin (Altoprev)


In [ ]:
prolog = "https://pubchem.ncbi.nlm.nih.gov/rest/pug"

str_cid = ",".join([ str(x) for x in cids])

url = prolog + "/compound/cid/" + str_cid + "/property/isomericsmiles/txt"
res = requests.get(url)
smiles = res.text.split()

print(smiles)

In [ ]:
mols = [ Chem.MolFromSmiles(x) for x in smiles ]
Chem.Draw.MolsToGridImage(mols, molsPerRow=4, subImgSize=(200,200), legends=[str(x) for x in cids] )

In [ ]:
# Radius set to 1 and 1024 bit size
from rdkit import DataStructs
from rdkit.Chem import rdFingerprintGenerator
mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=1,fpSize=1024)

fps = [ mfpgen.GetFingerprint(x) for x in mols ]


In [ ]:
# This calculates Tanimoto similarity for radius=1,fpSize=1024
for i in range(0, len(fps)) :
    for j in range(i+1, len(fps)) :
        
        score = DataStructs.TanimotoSimilarity(fps[i], fps[j])
        print(cids[i], "vs.", cids[j], ":", round(score,3), end='')
        
        if ( score >= 0.85 ):
            print(" ****")
        elif ( score >= 0.75 ):
            print(" ***")
        elif ( score >= 0.65 ):
            print(" **")
        elif ( score >= 0.55 ):
            print(" *")
        else:
            print(" ")

In [ ]:
# Radius set to 0 and 1024 bit size
mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=0,fpSize=1024)
fps0 = [ mfpgen.GetFingerprint(x) for x in mols ]


In [ ]:
# This calculates Tanimoto similarity for radius=0,fpSize=1024
for i in range(0, len(fps0)) :
    for j in range(i+1, len(fps0)) :
        
        score = DataStructs.TanimotoSimilarity(fps0[i], fps0[j])
        print(cids[i], "vs.", cids[j], ":", round(score,3), end='')
        
        if ( score >= 0.85 ):
            print(" ****")
        elif ( score >= 0.75 ):
            print(" ***")
        elif ( score >= 0.65 ):
            print(" **")
        elif ( score >= 0.55 ):
            print(" *")
        else:
            print(" ")

In [ ]:
# Radius set to 2 and 1024 bit size
mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2,fpSize=1024)
fps2 = [ mfpgen.GetFingerprint(x) for x in mols ]


In [ ]:
# This calculates Tanimoto similarity radius=2,fpSize=1024
for i in range(0, len(fps2)) :
    for j in range(i+1, len(fps2)) :
        
        score = DataStructs.TanimotoSimilarity(fps2[i], fps2[j])
        print(cids[i], "vs.", cids[j], ":", round(score,3), end='')
        
        if ( score >= 0.85 ):
            print(" ****")
        elif ( score >= 0.75 ):
            print(" ***")
        elif ( score >= 0.65 ):
            print(" **")
        elif ( score >= 0.55 ):
            print(" *")
        else:
            print(" ")

# Homework Part 2: Interpreting Data- PubChem Fingerprints

Run the code below to explore the statins molecules using PubChem Fingerprints. The does the following:

- Downloads the PubChem Fingerprint for the seven CIDs.
- Converts the downloaded fingerprints into bit vectors.
- Computes the pair-wise Tanimoto scores using the bit vectors.

**Turn in Part 2 as a written assignment.** 

I want you to compare these results to the MACCS keys in the Activity part of the lesson.

Describe any trends you observe. For example:
- Why doesn't the method of obtaining fingerprint data matter for final similarity calculations?
- Do similarity scores change depending on the number of fingerprint bits used?  
- If there is a difference, do the number of bits used give higher or lower values?


In [ ]:
# need user defined function o decode pubchem fingerprints
from base64 import b64decode
def PCFP_BitString(pcfp_base64) :

    pcfp_bitstring = "".join( ["{:08b}".format(x) for x in b64decode( pcfp_base64 )] )[32:913]
    return pcfp_bitstring

# Need statin CIDS
statin_cids = [    54454,  # Simvastatin (Zocor)
                   54687,  # Pravastatin (Pravachol)
                   60823,  # Atorvastatin (Lipitor)
                  446155,  # Fluvastatin (Lescol)   
                  446157,  # Rosuvastatin (Crestor)
                 5282452,  # Pitavastatin (Livalo)
                97938126 ] # Lovastatin (Altoprev)

# download each pubchem fingerprint, convert to bitstring and store in a list
# Method 1 with separate requests
statin_pcfp = []
for item in statin_cids:
    url = "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/CID/"+ str(item)+ "/property/Fingerprint2D/TXT"
    res = requests.get(url)
    text = res.text.rstrip("\n")
    pcfp_bitstring = PCFP_BitString(text)
    # use DataStructs.CreateFromBitString to convert a string to an RDKit bit vect object.
    statin_pcfp.append(DataStructs.CreateFromBitString(pcfp_bitstring))

# This calculates Tanimoto similarity of pubchem fingerprints
print("Method 1 download output")
for i in range(0, len(statin_pcfp)) :
    for j in range(i+1, len(statin_pcfp)) :
        
        score = DataStructs.TanimotoSimilarity(statin_pcfp[i], statin_pcfp[j])
        print(statin_cids[i], "vs.", statin_cids[j], ":", round(score,3), end='')
        
        if ( score >= 0.85 ):
            print(" ****")
        elif ( score >= 0.75 ):
            print(" ***")
        elif ( score >= 0.65 ):
            print(" **")
        elif ( score >= 0.55 ):
            print(" *")
        else:
            print(" ")
print()

# method 2 get Pubchem fingerprints in one request
# join to make one request
str_statin_cids = ",".join([ str(x) for x in statin_cids])
# generate the url
url = prolog + "/compound/cid/" + str_statin_cids + "/property/Fingerprint2D/TXT"
res = requests.get(url)

statin_pcfps_base64 = res.text.split()
statin_pcfps_bitstring = [ PCFP_BitString(x) for x in statin_pcfps_base64 ]

# use DataStructs.CreateFromBitString to convert a string to an RDKit bit vect object for calculations.
statin_pcfps_bitvect = [ DataStructs.CreateFromBitString(x) for x in statin_pcfps_bitstring ]

# This calculates Tanimoto similarity of pubchem fingerprints
print("Method 2 download output")
for i in range(0, len(statin_pcfps_bitvect)) :
    for j in range(i+1, len(statin_pcfps_bitvect)) :
        
        score = DataStructs.TanimotoSimilarity(statin_pcfps_bitvect[i], statin_pcfps_bitvect[j])
        print(statin_cids[i], "vs.", statin_cids[j], ":", round(score,3), end='')
        
        if ( score >= 0.85 ):
            print(" ****")
        elif ( score >= 0.75 ):
            print(" ***")
        elif ( score >= 0.65 ):
            print(" **")
        elif ( score >= 0.55 ):
            print(" *")
        else:
            print(" ")

# Homework Part 3: Interpreting Data- Comparing Distributions of Similarity Scores

**You will write a lab report for this Part**

Previously, you computed the similarity scores between some cholesterol-lowering drugs, and CID 60823 and CID 446155 had a Tanimoto score of **0.662**.  Based on the score distribution curve generated in the second section, we can say that the probablilty of two randomly selected compounds from PubChem having a Tanimoto score greater than 0.662 is **less than 1%**. This was with MACCS Keys. 

In this exercise, we will generate the distribution of the similarity scores among 1,000 compounds randomly selected from PubChem, using different molecular fingeprints and similarity metrics.

For molecular fingerprints, we will use the following:

- PubChem Fingerprint
- MACCS keys
- Morgan Fingerprint (ECFP4 analogue, 1024-bit-long)

For similarity metrics, we will use the following:

- Tanimoto similarity
- Dice similarity
- Cosine similarity

As a result, a total of 9 distribution curves will be generated.


**Lab report:**

1) Title, Author and Date
2) Introduction
    - Brief overview of molecular similarity and its role in cheminformatics
    - Define fingerprints (PubChem, MACCS, Morgan) and similarity metrics (Tanimoto, Dice, Cosine)
    - State the goal of the experiment
3) Methods: Describe how
    - CIDS were selected
    - SMILES were retrieved
    - fingerprints were generated
    - similarity scores were calculated
4) Results
    - Similarity Score Distributions (9 Histograms)
    - Top 1% Similarity Thresholds
|Fingerprint|Tanimoto|Dice|Cosine|
|-----------|--------|---------|---------|
|PubChem|---|---|---|
|MACCS|---|---|---|
|Morgan(radius = 2)|---|---|---|


5) Discussion:
    - Which fingerprints gave higher or lower similarity scores?
    - How did similarity metrics affect score distributions?
    - Did MACCS, PubChem or Morgan fingerprints behave differently? If so, why?
    - What cut-off would you use for virtual screening to find like molecules?


6) Conclusion:
   - summarize your findings in 2-3 sentences


**Run the following code cells (Step 1 and Step 2)**


**Step 1:** Generate 1,000 random CIDs, download the isomeric SMILES for them, and create the RDKit mol objects from the downloaded SMILES strings.

In [ ]:
#------------------------#
#  Generate 1,000 CIDs.  #
#------------------------#
import random
random.seed(2025)

cid_max = 121413818  # The maximum CID in PubChem as of May 2025

cids = []

for x in range(1000):
    cids.append(random.randint(1, cid_max + 1))
    
#-----------------------------------------#
#  Download SMILES strings from PubChem.  #
#-----------------------------------------#

chunk_size = 100

if len(cids) % chunk_size == 0 :
    num_chunks = int( len(cids) / chunk_size )
else :
    num_chunks = int( len(cids) / chunk_size ) + 1

smiles = []

for i in range(num_chunks):

    if (i == 0):
        print("Processing chunk ", end='')
    
    print(i, end=' ')
    
    idx1 = chunk_size * i
    idx2 = chunk_size * (i + 1)
    str_cids = ",".join([ str(x) for x in cids[idx1:idx2]])

    url = prolog + "/compound/cid/" + str_cids + "/property/isomericsmiles/txt"
    res = requests.get(url)

    if ( res.status_code == 200) :
        smiles.extend( res.text.split() )
    else :
        print("Chunk", i, "Failed to get SMILES.")
        
    time.sleep(0.2)

print("Done!")
print("# Number of SMILES : ", len(smiles))

mols = [ Chem.MolFromSmiles(x) for x in smiles ]


**Step 2:** Generate the fingerprints, compute the similarity scores, determine similarity thresholds, and make histograms.

In [ ]:
import matplotlib.pyplot as plt
from rdkit import DataStructs
from rdkit.Chem import MACCSkeys
%matplotlib inline

import sys
from base64 import b64decode

def PCFP_BitString(pcfp_base64) :

    pcfp_bitstring = "".join( ["{:08b}".format(x) for x in b64decode( pcfp_base64 )] )[32:913]
    return pcfp_bitstring
    
score_thresholds = []    # list to store the similarity score thrshold

fig = plt.figure(figsize=(12,12), dpi=300)
ax_idx = 0    # index for the panels in the figure

fp_types = [ 'pubchem', 'maccs', 'morgan' ]
metric_types = [ 'tanimoto', 'dice', 'cosine' ]


#-- Loop over each combination of fingerprints & metrics

for fp_idx in range(0, len(fp_types)) :    
    
    fps = []    #-- Must be re-initialized for every iteration!
    
    if ( fp_types[fp_idx] == 'pubchem' ) :    #-- download the PubChem fingerprints.     
        
        chunk_size = 100
        
        if len(cids) % chunk_size == 0 :
            num_chunks = int( len(cids) / chunk_size )
        else :
            num_chunks = int( len(cids) / chunk_size ) + 1

        fps_base64 = []


        for i in range(num_chunks):
    
            idx1 = chunk_size * i
            idx2 = chunk_size * (i + 1)
        
            str_cids = ",".join([ str(x) for x in cids[idx1:idx2]])
            url = prolog + "/compound/cid/" + str_cids + "/property/Fingerprint2D/TXT"
            res = requests.get(url)

            #print(res.url)
            fps_base64 = res.text.split()
            fps_bitstring = [ PCFP_BitString(x) for x in fps_base64 ]
        
            fps.extend([ DataStructs.CreateFromBitString(x) for x in fps_bitstring ])

            time.sleep(0.2)
    
    elif ( fp_types[fp_idx] == 'maccs') :    #-- Compute MACCS Keys. 
        
        fps = [ MACCSkeys.GenMACCSKeys(x) for x in mols if x != None ]
    
    elif ( fp_types[fp_idx] == 'morgan') :    #-- Compute Morgan fingerprints.    
    
        fps = rdFingerprintGenerator.GetMorganGenerator(radius=2,fpSize=1024)
        fps = [ mfpgen.GetFingerprint(x) for x in mols ]
    else:
        
        print("fp_type =", fp_type[fp_idx])
        sys.exit('Not implemented yet')

    mybins = [ x*0.01 for x in range(0,101) ]
    
    thresh_row = []
    
    for metric_idx in range(0,len(metric_types)) :
        
        print("Computing similarity scores: %s and %s" % (fp_types[fp_idx], metric_types[metric_idx]))

        scores = []

        for i in range(0, len(fps)) :
    
            for j in range(i+1, len(fps)) :
            
                if ( metric_types[metric_idx] == 'tanimoto'): 
                    
                    scores.append(DataStructs.TanimotoSimilarity(fps[i], fps[j]))
                                    
                elif ( metric_types[metric_idx] == 'dice'):
                    
                    scores.append(DataStructs.DiceSimilarity(fps[i], fps[j]))
                
                elif ( metric_types[metric_idx] == 'cosine'):
                    
                    scores.append(DataStructs.CosineSimilarity(fps[i], fps[j]))

        
        #-- Determine the score for top 1% pairs (99 percentile)
        
        scores.sort()
        thresh = scores[ round(len(scores) * 0.99) ]
        thresh_row.append( thresh )
        
        
        #-- Make a histogram for the current scores.
        
        ax_idx += 1
        
        plt.subplot(3, 3, ax_idx)
        plt.hist(scores, bins=mybins)
        plt.axvline(x=thresh, linestyle='--', color = 'r')
        mytitle = fp_types[fp_idx] + "-" + metric_types[metric_idx]
        plt.title( mytitle )
        
    score_thresholds.append(thresh_row)

print()
print('-' * 60)
print("FP","\t".join(metric_types), sep="\t")
print('-' * 60)
for i, row in enumerate(score_thresholds) :
    
    print(fp_types[i], "\t", sep='', end='')
    
    for j, col in enumerate(row) :
        print(round(score_thresholds[i][j],3), "\t", sep='', end='')
    
    print()